# TRACE-LAMPS Full Pipeline - Colab

Paper pipeline: Supervisor/Planner -> Package Acquisition -> Static Context Graph -> Semantic Classifier (fine-tuned CodeBERT) -> RC-PAA -> Critic/Verifier -> Decision/Audit.

Reasoning agents use Ollama Cloud `deepseek-v4-flash:cloud`. Classifier uses the existing fine-tuned CodeBERT `model.bin`.

In [ ]:
!pip -q install transformers ollama tqdm requests

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, re, shutil, sys

DRIVE_ROOT = Path('/content/drive/My Drive')
DRIVE_DATA = DRIVE_ROOT / 'NT230' / 'data'
ENV_PATH = DRIVE_DATA / '.env'
DRIVE_MODEL = DRIVE_DATA / 'd1' / 'saved_models' / 'checkpoint-best-acc' / 'model.bin'
LOCAL_MODEL = Path('/content/saved_models/checkpoint-best-acc/model.bin')
REPO_DIR = Path('/content/NT230')

def load_env(path):
    if not path.exists():
        return
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        m = re.match(r'^\$env:(\w+)\s*=\s*["\']?([^"\']+)["\']?', line)
        if m:
            os.environ.setdefault(m.group(1), m.group(2).strip())
            continue
        if '=' in line:
            k, _, v = line.partition('=')
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

load_env(ENV_PATH)
os.environ.setdefault('OLLAMA_MODEL', 'deepseek-v4-flash:cloud')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
!git clone --depth=1 https://github.com/khoilv2005/NT230.git /content/NT230
sys.path.insert(0, str(REPO_DIR / 'src'))

LOCAL_MODEL.parent.mkdir(parents=True, exist_ok=True)
assert DRIVE_MODEL.exists(), f'Missing CodeBERT checkpoint: {DRIVE_MODEL}'
shutil.copy(DRIVE_MODEL, LOCAL_MODEL)

assert os.getenv('OLLAMA_API_KEY'), 'Missing OLLAMA_API_KEY. Put it in NT230/data/.env or set os.environ manually.'
print('repo:', REPO_DIR)
print('model:', LOCAL_MODEL, round(LOCAL_MODEL.stat().st_size / 1e6, 2), 'MB')
print('ollama_model:', os.getenv('OLLAMA_MODEL'))

In [ ]:
from lamps.trace_pipeline import TraceLampsPipeline
from lamps.agents.classifier import ClassifierAgent

pipeline = TraceLampsPipeline(
    classifier=ClassifierAgent(checkpoint=LOCAL_MODEL, batch_size=64),
    use_ollama=True,
    ollama_model=os.getenv('OLLAMA_MODEL', 'deepseek-v4-flash:cloud'),
    package_threshold=0.72,
)
print('TRACE-LAMPS ready')

In [ ]:
# Edit package names here. Version can be None for latest.
package_specs = [
    ('requests', None),
    ('urllib3', None),
    ('certifi', None),
    ('bs4tools', '0.0.1'),
    ('grandslam', '0.1.0'),
]

In [ ]:
from tqdm.auto import tqdm
import json, time

OUT_DIR = DRIVE_DATA / 'trace_lamps_results'
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = OUT_DIR / 'package_results.jsonl'
BOARDS_PATH = OUT_DIR / 'evidence_boards.jsonl'
ERRORS_PATH = OUT_DIR / 'errors.jsonl'

def append_jsonl(path, obj):
    with path.open('a', encoding='utf-8') as f:
        f.write(json.dumps(obj, ensure_ascii=False) + '\n')

for p in [RESULTS_PATH, BOARDS_PATH, ERRORS_PATH]:
    if p.exists():
        p.unlink()

success = 0
errors = 0
for package, version in tqdm(package_specs, desc='TRACE-LAMPS live'):
    started = time.time()
    try:
        result = pipeline.analyze_package(package, version=version)
        record = result.to_dict()
        record['runtime_seconds'] = round(time.time() - started, 2)
        append_jsonl(RESULTS_PATH, record['verdict'])
        append_jsonl(BOARDS_PATH, record['evidence_board'])
        success += 1
        print(package, '=>', result.verdict.label, 'confidence=', round(result.verdict.confidence, 4), 'trigger=', result.verdict.trigger_file)
    except Exception as exc:
        errors += 1
        err = {'package': package, 'version': version, 'error': repr(exc)}
        append_jsonl(ERRORS_PATH, err)
        print('ERROR', package, repr(exc))

summary = {
    'pipeline': 'TRACE-LAMPS: Planner -> Acquisition -> Context Graph -> CodeBERT -> RC-PAA -> Verifier -> Audit',
    'ollama_model': os.getenv('OLLAMA_MODEL', 'deepseek-v4-flash:cloud'),
    'classifier': 'fine-tuned CodeBERT model.bin',
    'n_requested': len(package_specs),
    'n_success': success,
    'n_errors': errors,
    'results_path': str(RESULTS_PATH),
    'evidence_boards_path': str(BOARDS_PATH),
    'errors_path': str(ERRORS_PATH),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))